# StatsWales: dimensions, codes and suppressed values

These are **synthetic teaching responses, not published statistics**. The checked fixtures use each provider’s response shape and exercise SSDU’s actual acquisition, paging, rate-limit accounting and transformation code without network access. Run from this repository with `pip install -e ".[dev]"`; CI executes each notebook in a fresh directory and blocks networking. For published data, omit `client=client` and choose selectors after inspecting provider metadata. Keep the same Store, Request and Dataset API.

In [ ]:
import sys
from pathlib import Path

from ssdu import Request, Store

# Jupyter may start the kernel in this notebook's directory.
for checkout in (Path.cwd(), *Path.cwd().parents):
    if (checkout / "examples" / "dataset_contract.py").is_file():
        sys.path.insert(0, str(checkout))
        break
from examples.dataset_contract import check_dataset, fixture_client, no_network

store = Store(Path.cwd() / "store")

## One request and one retrieval API

Native selectors differ; inspect them through source metadata. Retrieval, caching and dataset access are shared.

In [ ]:
request = Request(
    provider="statswales",
    dataset_id="40ddfa33-9ba8-4278-ac27-1f1b0bc5fc0f",
    params={"filters": [{"AreaID": ["001", "002", "003"]}, {"YearID": ["2025"]}]},
    page_size=2,
    max_rows=10,
)
with no_network(), fixture_client("statswales") as client:
    dataset = store.fetch_dataset(request, client=client)
    fixture_http_attempts = client.attempts
assert fixture_http_attempts >= 0  # Zero on a rerun means the cache was reused.
assert dataset.snapshot_id is not None
print(dataset.snapshot_id)

## The standard observation table

Area labels are linked to source dimension codes. Padding and exact values stay in `value_raw`; `*` stays null and Notes are preserved. A missing unit is explicitly unknown, so this fixture should not be described as a count or percentage based on its title.

In [ ]:
observations = dataset.to_arrow("observations")
frame = dataset.to_pandas("observations")
check_dataset(dataset, "statswales")
assert len(frame) == observations.num_rows
frame

## Machine-readable metadata and retained source records

Inspect coverage, release, variable definitions, validation and transformation provenance before interpreting values.

In [ ]:
descriptor = dataset.contract
assert descriptor["quality"]["source_rows"] == 3
assert descriptor["quality"]["observation_rows"] == observations.num_rows
assert dataset.to_arrow("source").num_rows == 3
descriptor

## Cache reuse and explicit offline replay

Pin the snapshot ID to reopen the same transformed bytes independently of provider updates. `offline=True` forbids acquiring missing source responses.

In [ ]:
with no_network():
    cached = store.fetch_dataset(request, offline=True)
    reopened = store.open_dataset(dataset.snapshot_id)
assert cached.snapshot_id == dataset.snapshot_id
assert reopened.to_arrow().equals(observations)
assert reopened.contract == descriptor
print({"offline_replay": True, "fixture_http_attempts": fixture_http_attempts})

The executable `examples/dataset_contract.py` combines all three providers into a pinned collection, exports it, imports it into a fresh store and verifies the same dataset contracts and observation tables offline.